## Setup runner

In [ ]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="xr client utilities example")
imd_runner.load(0)

In [ ]:
def notify_all(message):
    for command in imd_runner.app_server.commands:
        if "/notify" in command:
            imd_runner.app_server.run_command(command, dict(message=message))

In [ ]:
import numpy as np
from follower import Follower, Group
from nanover.imd import ParticleInteraction


CHECKPOINT = None
FOLLOWER: Follower | None = None
GROUPS: list[Group] = []


def clone_omni_omm(omni_omm):
    """
    Create an independent copy of a given NanoVer OpenMMSimulation.
    """
    from nanover.openmm import serializer

    # get the underlying openmm simulation object
    omm = omni_omm.simulation

    # clone it by serializing and then deserializing
    data = serializer.serialize_simulation(omm)
    omm_clone = serializer.deserialize_simulation(data)

    # wrap the cloned openmm simulation in OpenMMSimulation
    omni_omm_clone = OpenMMSimulation.from_simulation(omm_clone)

    # give it different name
    omni_omm_clone.name = omni_omm.name + "'"

    return omni_omm_clone


def set_checkpoint():
    global CHECKPOINT, GROUPS
    CHECKPOINT = clone_omni_omm(imd_runner.simulation)
    GROUPS = []
    notify_all("CHECKPOINT SET")


def reset_to_checkpoint():
    global FOLLOWER
    if FOLLOWER is not None:
        FOLLOWER.stop()
        FOLLOWER = None
    imd_runner.simulations[0] = CHECKPOINT
    imd_runner.load(0)


def replay_from_checkpoint():
    global FOLLOWER

    if CHECKPOINT is None:
        notify_all("NO CHECKPOINT")
    else:
        reset_to_checkpoint()
        notify_all("REPLAYING FROM CHECKPOINT")
        FOLLOWER = Follower.from_runner(imd_runner)
        FOLLOWER.start(GROUPS)


def on_interaction_updated(*, key: str, interaction: ParticleInteraction):
    if CHECKPOINT is not None and FOLLOWER is None:
        for group in GROUPS:
            if set(group.particles) == set(interaction.particles):
                positions = imd_runner.app_server.frame_publisher.current_frame.particle_positions
                centroid = np.average(positions[interaction.particles], axis=0)
                group.path.add_point(centroid)
                break
        else:
            GROUPS.append(Group(particles=interaction.particles))

imd_runner.app_server.imd.interaction_updated.add_callback(on_interaction_updated)

# add the command to the server,
# commands with names beginning with "user/" are displayed in the VR client
imd_runner.app_server.register_command("user/checkpoint/set", set_checkpoint)
imd_runner.app_server.register_command("user/checkpoint/reset", reset_to_checkpoint)
imd_runner.app_server.register_command("user/checkpoint/replay", replay_from_checkpoint)